# H-Beauty: catalog photos for the new looks

The salon can't photograph new looks, so **FLUX 3 Image Klein** draws each one onto the catalog's bare
portrait (`no-makeup.webp`), from a written description of its makeup, on Kaggle's free GPU. You get **4 versions of
each look**; the artist picks the best one, and that becomes the look's catalog photo («مرجع» on the bench).

The looks: those waiting in `src/data/new-looks.json` (none now: the season's six were drawn with ChatGPT).

**Settings (right panel, once):**
1. *Session options* → **Accelerator: GPU T4 x2**, **Internet: on** (needs a phone-verified Kaggle account).
2. *Add Input* → your dataset made from the **new** `hb-gpu-kit.zip` (upload it as a new version of the dataset).
3. *Add-ons → Secrets*: **HF_TOKEN** (your Hugging Face Read token; accept the terms on the
   `black-forest-labs/flux-3-image-klein` page first). Switch it on for this notebook.

Then **Save Version → Save & Run All**. It runs in the background (about 1–2 hours); you can close the page.
When it's done, open the version → *Output* → download **new-looks.zip** and send it in the chat.


In [ ]:
# 1. Find the kit, read the secret, install (2-3 min)
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/scripts/gpu/new_looks.py', recursive=True)
assert hits, 'Add the NEW hb-gpu-kit dataset (it has scripts/gpu/new_looks.py): right panel -> Add Input'
shutil.copytree(hits[0].split('/scripts/gpu/')[0], '/kaggle/working/kit', dirs_exist_ok=True)

from kaggle_secrets import UserSecretsClient
try: os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception: os.environ['HF_TOKEN'] = ''
assert os.environ['HF_TOKEN'], 'Add the HF_TOKEN secret (Add-ons -> Secrets) and switch it on for this notebook'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!pip install -q -U diffusers transformers accelerate huggingface_hub hf_transfer mediapipe opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# 2. Draw the looks (about 1-2 hours): each finished picture prints a line
VARIANTS = 4   # versions per look for the artist to choose from
!cd /kaggle/working && python kit/scripts/gpu/new_looks.py --variants {VARIANTS} --max-minutes 600 2>&1 | grep -v -E 'W0000|I0000|XNNPACK|feedback tensors|Exception ignored|TypeError: .NoneType|File "/usr'


In [ ]:
# 3. The sheets: before, then the versions (the steadiest face first). Full pictures are in new-looks.zip under Output.
from IPython.display import Image, display
for f in sorted(glob.glob('/kaggle/working/new-looks/sheet_*.jpg')):
    print(f.split('/')[-1]); display(Image(filename=f, width=1200))
